<div class="alert alert-block alert-info">

# Part 4: Machine Learning: Loading data for Training and Test Setes and Creating Decision Tree Models

### Decision Tree

Decision Tree Classification is a machine learning algorithm that predicts whether a molecule is active or inactive by asking a series of yes/no questions based on its features. We will employ this method to look for MACCS Keys. The algorithm builds a tree-like model where each question splits the data to better separate active from inactive compounds. It chooses the questions based on how well they divide the data into pure groups. When a group is pure, it will mostly contain molecules of one class. 

Previously we saved the downsampled data sets(training and test) with 163 MACCS Keys (zero variance removed) for use later. Let's reload them to create our training and test sets.

In [ ]:
import numpy as np                                   # load numpy library

X_train = np.load("X_train_downsampled_MACCS163.npy")   # downsampled balanced set MACCS keys 163 values
y_train = np.load("y_train_downsampled_MACCS163.npy")   # downsampled balanced set activity values
X_test = np.load("X_test_MACCS163.npy")                 # test set MACCS keys 163 values
y_test = np.load("y_test163.npy")                       # test set activity values

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

Load necessary machine learning libraries

In [ ]:
from sklearn.tree import DecisionTreeClassifier    #-- Decision Tree
from sklearn.metrics import classification_report #provides detailed report that includes precision and sensitivity
from sklearn.metrics import confusion_matrix      # gives a 2x2 matrix for true netatives, false positives, false negatives and true positives
from sklearn.metrics import accuracy_score        # computes accuracy = number of correct preictions/total number of preditions
from sklearn.metrics import roc_auc_score         # Computs Area under the ROC curve, evaluates trade-off of true positive rate and false positive rate

In [ ]:
# set up the DT classification model
clf_DT = DecisionTreeClassifier( random_state=0 )    

In [ ]:
# Train the model by fitting it to the data (using the default values for all parameters)
# we are using the same training data as before
clf_DT.fit( X_train ,y_train )    

In [ ]:
# Apply the model to predict the training compound's activity.
y_true, y_pred = y_train, clf_DT.predict( X_train )    

In [ ]:
# generate confusion matrix
CMat = confusion_matrix( y_true, y_pred )   
print(CMat)    # [[TN, FP], 
               #  [FN, TP]]
# Extracting TN, FP, FN, TP from the confusion matrix               
TN = CMat[0, 0]  # True Negatives
FP = CMat[0, 1]  # False Positives
FN = CMat[1, 0]  # False Negatives
TP = CMat[1, 1]  # True Positives

print("True Negatives (TN):", TN)
print("False Positives (FP):", FP)
print("False Negatives (FN):", FN)
print("True Positives (TP):", TP)

print("Total predictions:", TN + FP + FN + TP)

In [ ]:
acc  = accuracy_score( y_true, y_pred ) #correct predictions/total predictions = (TP+TN)/(TP+TN+FP+FN)
prec = TP / (TP + FP)    # precision = TP / (TP + FP) measures how well model identifies actual positives
sens = TP / (FN + TP)    # sensitivity = TP / (FN + TP) measures how well model identifies actual positives
spec = TN / (TN + FP)    # specificity = TN / (TN + FP)measures how well model identifies actual negatives
bacc = (sens + spec) / 2    #averages sensititivy and specificity
f1_score = 2 * (prec * sens) / (prec + sens)  # F1 score is the harmonic mean of precision and sensitivity

y_score = clf_DT.predict_proba( X_train )[:, 1] #returns proability of each class, [:, 1] extracts the probability of the positive class (label 1) for each sample.
auc = roc_auc_score( y_true, y_score ) #measures how well the model ranks psitive vs negative samples. AUC = 1.0 → perfect model; AUC = 0.5 → random guessing.

print("Training set performance metrics:")
print(f"Accuracy          = {acc:.4f}") # how accurate is the model overall
print(f"Precision         = {prec:.4f}") # How well it identifies actual positives (precision)
print(f"Sensitivity       = {sens:.4f}") # How well it catches positives (sensitivity)
print(f"Specificity       = {spec:.4f}") # How well it avoids false negatives (specificity)
print(f"Balanced Accuracy = {bacc:.4f}") # How balanced its performance is across classes
print(f"F1 Score          = {f1_score:.4f}") # Harmonic mean of precision and sensitivity
print(f"AUC-ROC           = {auc:.4f}")  #How well it ranks predictions (AUC)

<div class="alert alert-block alert-warning">
<strong>Interpreting the confusion matrix</strong> Write your answers in the following raw cell.

1) How does the sum of the predictions relate to number of molecules the training set? 

2) How many actives are in the training set? How many would a perfect model predict? Is this predicting more or less?

3) How many inactives are in the training set? How many would a perfect model predict? Is this predicting more or less?

4) Is this model perfect? Is it *"good enough"*?

5) Compared to your results from naïve Bayes (last activity), does this look to be a better model for predictions?

<div class="alert alert-block alert-warning"> Write your answers in the following raw cell.
<strong>Interpreting key metrics from the confusion matrix</strong>

Using the Classification Metrics Interpretation Guide, how well did the decision tree model predict those molecules in the training set?

<div class="alert alert-block alert-info">
<details>
<summary>Classification Metrics Interpretation Guide</summary>


| **Metric**| **What It Measures** | **How to Interpret the Value** |
|:--------|:--------|:--------|
|  **Accuracy**   | Overall % of correct predictions | ✅ ≥ 0.80 = High accuracy<br>⚠️ 0.60–0.79 = Moderate *Can be misleading with class imbalance* <br>🔴 < 0.60 = Poor accuracy|
|  **Balanced Accuracy**   |  Avg. of sensitivity (recall) and specificity  |  ✅ ≥ 0.80 = Strong<br>⚠️ 0.60–0.79 = Moderate<br>🔴 < 0.60 = Poor|
|  **Sensitivity** (Recall)  |  % of actual actives correctly predicted (TPR / Recall)   | ✅ ≥ 0.80 = Few missed positives<br>⚠️ 0.60–0.80 = Moderate<br>🔴 < 0.60 = Many positives missed  |
|  **Specificity**   |  % of actual inactives correctly predicted (TNR)  |  ✅ ≥ 0.80 = Few false positives<br>⚠️ 0.60–0.79 = Moderate<br>🔴 < 0.60 = Too many false positives  |
|  **Precision**   |  % of predicted positives that are truly positive  |  ✅ ≥ 0.80 = Few false positives<br>⚠️ 0.60–0.79 = Moderate<br>🔴 < 0.60 = Too many false positives  |
|**F1-score**| harmonic Mean of precision and recall|✅ ≥ 0.75 = Few false Positives<br>⚠️ 0.60–0.79 = Moderate<br>🔴 < 0.60 = Too many false positives
|  **AUC-ROC**|  Ability to rank actives above inactives across thresholds |  ✅ ≥ 0.80 = Strong discrimination<br>⚠️ 0.70–0.79 = Acceptable<br>🔴 < 0.70 = Weak model<br>🚫 ~0.50 = Random guessing |

**Quick Rules of Thumb:**
- If accuracy is high but balanced accuracy is low → suspect class imbalance.
- Use balanced accuracy when your dataset has a lot more inactives than actives.
- Sensitivity is important when missing an active could be costly (e.g., in drug screening).
- Specificity is important when false positives are costly (e.g., expensive follow-up experiments).
- AUC-ROC gives a broader view of model quality — useful even when you're not picking a classification threshold yet.

Let's test the molecules in the test set.

In [ ]:
y_true, y_pred = y_test, clf_DT.predict(X_test)    #-- Apply the model to predict the test set compounds' activity.

In [ ]:
# generate confusion matrix
CMat = confusion_matrix( y_true, y_pred )   
print(CMat)    # [[TN, FP], 
               #  [FN, TP]]
# Extracting TN, FP, FN, TP from the confusion matrix               
TN = CMat[0, 0]  # True Negatives
FP = CMat[0, 1]  # False Positives
FN = CMat[1, 0]  # False Negatives
TP = CMat[1, 1]  # True Positives

print("True Negatives (TN):", TN)
print("False Positives (FP):", FP)
print("False Negatives (FN):", FN)
print("True Positives (TP):", TP)

print("Total predictions:", TN + FP + FN + TP)

In [ ]:
acc  = accuracy_score( y_true, y_pred ) #correct predictions/total predictions = (TP+TN)/(TP+TN+FP+FN)
prec = TP / (TP + FP)    # precision = TP / (TP + FP) measures how well model identifies actual positives
sens = TP / (FN + TP)    # sensitivity = TP / (FN + TP) measures how well model identifies actual positives
spec = TN / (TN + FP)    # specificity = TN / (TN + FP)measures how well model identifies actual negatives
bacc = (sens + spec) / 2    #averages sensititivy and specificity
f1_score = 2 * (prec * sens) / (prec + sens)  # F1 score is the harmonic mean of precision and sensitivity

y_score = clf_DT.predict_proba( X_test )[:, 1] #returns proability of each class, [:, 1] extracts the probability of the positive class (label 1) for each sample.
auc = roc_auc_score( y_true, y_score ) #measures how well the model ranks psitive vs negative samples. AUC = 1.0 → perfect model; AUC = 0.5 → random guessing.

print("Training set performance metrics:")
print(f"Accuracy          = {acc:.4f}") # how accurate is the model overall
print(f"Precision         = {prec:.4f}") # How well it identifies actual positives (precision)
print(f"Sensitivity       = {sens:.4f}") # How well it catches positives (sensitivity)
print(f"Specificity       = {spec:.4f}") # How well it avoids false negatives (specificity)
print(f"Balanced Accuracy = {bacc:.4f}") # How balanced its performance is across classes
print(f"F1 Score          = {f1_score:.4f}") # Harmonic mean of precision and sensitivity
print(f"AUC-ROC           = {auc:.4f}")  #How well it ranks predictions (AUC)

<div class="alert alert-block alert-warning">
<strong>Interpreting key metrics from the confusion matrix</strong> Write your answer in the following raw cell.

1) What is the purpose of the test set?

2) Using the Classification Metrics Interpretation Guide, how well did the decision tree model predict those molecules in the test set?

3) Does this still look like a model that does a better job than the naïve Bayes (last activity) model? Explain.

Such high performance on the training set, especially for a flexible model like a decision tree,  is often a sign of **overfitting**.

When applied to predict the activity of the **training** compounds, the DT classifier resulted in very high scores (>0.99) for all five performance measures considered here. The model may have memorized the training data, especially if the tree is very deep or was not pruned. This would result in poor generalization to new, unseen data. As we saw in the test set, the model performed poorly, which is why **test set** evaluation is critical.

## Model building through cross-validation

In the above section, the models were developed using the default values for many optional hyperparameters, which cannot be learned by the training algorithm.  For example, when building a **decision tree** model, one should specify how deep the *tree* should be, how many compounds should be allowed in a *single leaf*, what is the minimum number of compounds in a *single leaf*, etc.

Grid search and cross-validation are techniques used together to find the best settings for a machine learning model and ensure it generalizes well to new data. Decision trees have additional settings called hyperparameters. These are values that control how the model learns (e.g., the maximum depth of a decision tree, or how many samples are needed to split a node). These are not learned from the data but must be chosen manually. **Grid search** is a method that systematically tries different combinations of hyperparameter values to see which perform best.

To avoid relying on just one train/test split, cross-validation divides the training data into several folds (often 5 or 10), training the model on some folds and validating on the rest, rotating through all folds. This gives a more reliable estimate of model performance. When used together, grid search with cross-validation helps find the best hyperparameter combination while reducing the risk of overfitting.

The cells below demonstrate how to perform hyperparameter optimization through 10-fold cross-validation.  In this example, five values for each of three hyperparameters used in the decision tree are considered (max_depth, min_samples_split, and min_samples_leaf), resulting in a total of 125 combination of the parameter values (= 5 x 5 x 5).  For each combination, 10 models are generated (through 10-fold cross validation) and the average performance will be tracked.  The goal is to find the parameter value combination that results in the highest average performance score (e.g., 'roc_auc') from the 10-fold cross validation.

In [ ]:
#GridSearchCV stands for Grid Search with Cross-Validation.
from sklearn.model_selection import GridSearchCV 

In [ ]:
#For each model configuration, calculate both ROC-AUC and balanced accuracy
scores = [ 'roc_auc', 'balanced_accuracy' ] 

In [ ]:
ncvs = 10  #sets number of cross-validation folds to 10, the training set is split into 10 parts. In each iteration, the model is trained on 9 parts and validated on the remaining 1.
                        # np.linspace() to create evenly spaced integer values in the given ranges.
max_depth_range         = np.linspace( 3, 7, num=5, dtype='int32' )  #Maximum depth of the tree (e.g., [3, 4, 5, 6, 7])
min_samples_split_range = np.linspace( 3, 7, num=5, dtype='int32' )  # Minimum number of samples needed to split a node
min_samples_leaf_range  = np.linspace( 2, 6, num=5, dtype='int32' )  #Minimum number of samples needed to be a leaf node

param_grid = dict( max_depth=max_depth_range,                     #This is a dictionary mapping parameter names to lists of values to try.
                   min_samples_split=min_samples_split_range,     #GridSearchCV will try every combination of the 5 × 5 × 5 = 125 settings.
                   min_samples_leaf=min_samples_leaf_range )

clf_DT_CV = GridSearchCV( DecisionTreeClassifier( random_state=0 ),     #creates base model to optimize on AUC and balanced accuracy
                    param_grid=param_grid, cv=ncvs, scoring=scores, refit='roc_auc',
                    return_train_score = True)   

In [ ]:
# This cell will take some time to run. In testing it took anywhere from 15 seconds over a 1 minute, but that will
# depend on user hardware and what is stored in RAM.
clf_DT_CV.fit( X_train, y_train )   #triggers the full grid search with cross-validation
print("Best parameter set", clf_DT_CV.best_params_) #prints the best-performing hyperparameters. Tells us which combo gives highest CV and used for future predict.

If necessary, it is possible to look into the performance data for each parameter value combination (stored in **clf.cv_results_**), as shown in the following cell.

In [ ]:
means_1a = clf_DT_CV.cv_results_['mean_train_roc_auc'] #Gives mean and STDV for AUC for training data 
stds_1a  = clf_DT_CV.cv_results_['std_train_roc_auc']

means_1b = clf_DT_CV.cv_results_['mean_test_roc_auc'] #Gives mean and STDV for AUC for test data 
stds_1b  = clf_DT_CV.cv_results_['std_test_roc_auc']

means_2a = clf_DT_CV.cv_results_['mean_train_balanced_accuracy'] #Gives mean and STDV for Balanced accuracy for training data 
stds_2a  = clf_DT_CV.cv_results_['std_train_balanced_accuracy']

means_2b = clf_DT_CV.cv_results_['mean_test_balanced_accuracy']  #Gives mean and STDV for Balanced accuracy  for test data 
stds_2b  = clf_DT_CV.cv_results_['std_test_balanced_accuracy']

iterobjs = zip( means_1a, stds_1a, means_1b, stds_1b,           #Combines all the relevant stats with their corresponding parameter combinations into one iterable.
                means_2a, stds_2a, means_2b, stds_2b, clf_DT_CV.cv_results_['params'] )

for m1a, s1a, m1b, s1b, m2a, s2a, m2b, s2b, params in iterobjs :  #for each grid point gives all the data

    print( "Grid %r : %0.4f %0.04f %0.4f %0.04f %0.4f %0.04f %0.4f %0.04f"
           % ( params, m1a, s1a, m1b, s1b, m2a, s2a, m2b, s2b))  

Uncomment the following cell to look into additional performance data stored in cv_result_.

In [ ]:
#print(clf_DT_CV.cv_results_)

It is important to understand that each model built through 10-fold cross-validation during hyperparameter optimization uses only 90% of the compounds in the training set and the remaining 10% is used for testing that model.  After all parameter value combinations are evaluated, the best parameter values are selected and used to rebuild a model from **all** compounds in the training set.  **GridSearchCV()** takes care of this last step automatically.  Therefore, there is no need to take an extra step to build a model using **cls.fit()** after hyperparameter optimization.

In [ ]:
# Apply the model to predict the training compound's activity.
y_true, y_pred = y_train, clf_DT_CV.predict( X_train )    

In [ ]:
# generate confusion matrix
CMat = confusion_matrix( y_true, y_pred )   
print(CMat)    # [[TN, FP], 
               #  [FN, TP]]
# Extracting TN, FP, FN, TP from the confusion matrix               
TN = CMat[0, 0]  # True Negatives
FP = CMat[0, 1]  # False Positives
FN = CMat[1, 0]  # False Negatives
TP = CMat[1, 1]  # True Positives

print("True Negatives (TN):", TN)
print("False Positives (FP):", FP)
print("False Negatives (FN):", FN)
print("True Positives (TP):", TP)

print("Total predictions:", TN + FP + FN + TP)

In [ ]:
acc  = accuracy_score( y_true, y_pred ) #correct predictions/total predictions = (TP+TN)/(TP+TN+FP+FN)
prec = TP / (TP + FP)    # precision = TP / (TP + FP) measures how well model identifies actual positives
sens = TP / (FN + TP)    # sensitivity = TP / (FN + TP) measures how well model identifies actual positives
spec = TN / (TN + FP)    # specificity = TN / (TN + FP)measures how well model identifies actual negatives
bacc = (sens + spec) / 2    #averages sensititivy and specificity
f1_score = 2 * (prec * sens) / (prec + sens)  # F1 score is the harmonic mean of precision and sensitivity

y_score = clf_DT_CV.predict_proba( X_train )[:, 1] #returns proability of each class, [:, 1] extracts the probability of the positive class (label 1) for each sample.
auc = roc_auc_score( y_true, y_score ) #measures how well the model ranks psitive vs negative samples. AUC = 1.0 → perfect model; AUC = 0.5 → random guessing.

print("Training set performance metrics:")
print(f"Accuracy          = {acc:.4f}") # how accurate is the model overall
print(f"Precision         = {prec:.4f}") # How well it identifies actual positives (precision)
print(f"Sensitivity       = {sens:.4f}") # How well it catches positives (sensitivity)
print(f"Specificity       = {spec:.4f}") # How well it avoids false negatives (specificity)
print(f"Balanced Accuracy = {bacc:.4f}") # How balanced its performance is across classes
print(f"F1 Score          = {f1_score:.4f}") # Harmonic mean of precision and sensitivity
print(f"AUC-ROC           = {auc:.4f}")  #How well it ranks predictions (AUC)

<div class="alert alert-block alert-warning">
<strong>Interpreting key metrics from the confusion matrix on training set</strong> Write your answer in the following raw cell.

Compare these performance data with those from the original decision tree. (for the training set only). Does this indicate any overfitting?  

In [ ]:
#-- Apply the model to predict the test set compounds' activity.
y_true, y_pred = y_test, clf_DT_CV.predict(X_test)    

In [ ]:
# generate confusion matrix
CMat = confusion_matrix( y_true, y_pred )   
print(CMat)    # [[TN, FP], 
               #  [FN, TP]]
# Extracting TN, FP, FN, TP from the confusion matrix               
TN = CMat[0, 0]  # True Negatives
FP = CMat[0, 1]  # False Positives
FN = CMat[1, 0]  # False Negatives
TP = CMat[1, 1]  # True Positives

print("True Negatives (TN):", TN)
print("False Positives (FP):", FP)
print("False Negatives (FN):", FN)
print("True Positives (TP):", TP)

print("Total predictions:", TN + FP + FN + TP)

In [ ]:
acc  = accuracy_score( y_true, y_pred ) #correct predictions/total predictions = (TP+TN)/(TP+TN+FP+FN)
prec = TP / (TP + FP)    # precision = TP / (TP + FP) measures how well model identifies actual positives
sens = TP / (FN + TP)    # sensitivity = TP / (FN + TP) measures how well model identifies actual positives
spec = TN / (TN + FP)    # specificity = TN / (TN + FP)measures how well model identifies actual negatives
bacc = (sens + spec) / 2    #averages sensititivy and specificity
f1_score = 2 * (prec * sens) / (prec + sens)  # F1 score is the harmonic mean of precision and sensitivity

y_score = clf_DT_CV.predict_proba( X_test )[:, 1] #returns proability of each class, [:, 1] extracts the probability of the positive class (label 1) for each sample.
auc = roc_auc_score( y_true, y_score ) #measures how well the model ranks psitive vs negative samples. AUC = 1.0 → perfect model; AUC = 0.5 → random guessing.

print("Training set performance metrics:")
print(f"Accuracy          = {acc:.4f}") # how accurate is the model overall
print(f"Precision         = {prec:.4f}") # How well it identifies actual positives (precision)
print(f"Sensitivity       = {sens:.4f}") # How well it catches positives (sensitivity)
print(f"Specificity       = {spec:.4f}") # How well it avoids false negatives (specificity)
print(f"Balanced Accuracy = {bacc:.4f}") # How balanced its performance is across classes
print(f"F1 Score          = {f1_score:.4f}") # Harmonic mean of precision and sensitivity
print(f"AUC-ROC           = {auc:.4f}")  #How well it ranks predictions (AUC)

<div class="alert alert-block alert-warning">
<strong>Interpreting key metrics from the confusion matrix on test set</strong> Write you answer in the following raw cell.

Compare these performance data with those from the original decision tree. (for the test set only).  Has prediction performace improved?


Let's use the new cross-validated decision tree model to predict the same molecules we had used before.

In [ ]:
# load necessary libraries
from rdkit import Chem
from rdkit.DataStructs import ConvertToNumpyArray
from rdkit.Chem import MACCSkeys
import numpy as np

In [ ]:
#Define new SMILES string
new_smiles = "CC(=O)OC1=CC=CC=C1C(=O)O" # CID 2242 aspirin should be INactive
#new_smiles = "C1=CC=C(C=C1)C(C2=CC=CC=C2)(C3=CC=CC=C3Cl)N4C=CN=C4" # CID = 2812 should be active
#new_smiles = "C1=CC=C(C(=C1)C2=NC(=NO2)C3=CC=NC=C3)Cl" #CID 65758 should be active
#new_smiles = "C1=CC(=CC=C1C2=COC3=CC(=CC(=C3C2=O)O)O)O" #CID 5280961 should be active
#new_smiles = "CN(C1CCN(CC1)C2=NC3=CC=CC=C3N2CC4=CC=C(C=C4)F)C5=NC=CC(=O)N5" #CID 65906 should be INactive
#new_smiles = "C1=CNC(=O)NC1=O" #CID 1174 should be INactive
#new_smiles = "CCCCCC1=CC(=C2C=CC(OC2=C1)(C)CCC=C(C)C)O" #CID30219 not in datbase (CBC)
#new_smiles = "C[C@H]1C[C@@H](C(=O)[C@@H](C1)[C@@H](CC2CC(=O)NC(=O)C2)O)C" #CID 6197 should be active
#new_smiles = "CCCCCC1=CC(=C2[C@@H]3C=C(CC[C@H]3C(OC2=C1)(C)C)C)O" #CID16078 in database and should be active (THC)
#new_smiles = "COC1=CC(=CC(=C1OC)OC)CCN" #CID4076 not in database (mescaline)
#new_smiles = "CC1=C(C(CCC1)(C)C)/C=C/C(=C/C=C/C(=C/CO)/C)/C" # vitamin A not in database
mol = Chem.MolFromSmiles(new_smiles)
mol

In [ ]:
fp = MACCSkeys.GenMACCSKeys(mol)
# Drop the MACCS Keys that were dropped in generation of the model. In our case those were 0, 1, 2, and 4
# create a set of the values in the fingerprint we want to drop
MACCS_to_drop = {0,1,2,4} # use a set for faster lookup

# Keep items if their index is NOT in the removal set
fpdrop =[item for i, item in enumerate(fp) if i not in MACCS_to_drop]

X_query = np.array(fpdrop).reshape(1, -1) # The classifier needs a 2D shape, but we have 1D list. We reshape (1, n_bits)
prediction = clf_DT_CV.predict(X_query)

if prediction == 0:
    print("Molecule is inactive for human aromatase.")
else:
    print("Molecule is active for human aromatase. Active may be agonist or antagonist in this model.")


#### Using the saved selector
In the previous code cell we had to tell the program to drop specific MACCS Keys to go from 163 to 167. This required us to remember which features had zerro variance. Recall that we also saved our variance selector as well to get around this issue. We can use that here again.

In [ ]:
import joblib
# Load your saved variance selector
sel = joblib.load("maccs_variance_selector.joblib")

In [ ]:
# Use the saved variance selector to drop the four MACCS that had zero variance and predict activity
arr = np.zeros((1,), dtype=int) 
ConvertToNumpyArray(fp, arr)
arr_filtered = sel.transform(arr.reshape(1, -1))

prediction = clf_DT_CV.predict(arr_filtered)

print(prediction)
if prediction ==0:
    print("Molecule is inactive.")
else:
    print("Molecule is active.")

<div class="alert alert-block alert-warning">
<strong>Comparing Naive Bayes and Cross Validated Decision Tree Models</strong> Write your answers in the folloing raw cell.

1) Recall in the Naive Bayes model, CIDS 65906 and 6197 were predicted to have the opposite activity. Does this cross validated decision tree do a better job at predicting them?

2) How do these to models compare in their accuracy and AUC-ROC? Is one particularly better than the other in the test set data? 



#### Creating a pipeline for the Cross Validated Decision Tree Model

Now that we have a cross-validated decision tree model that works, we can create a pipeline that can be saved and reloaded later for predicting the activity of new molecules.

Rather than using the training and test sets that were already filtered to remove MACCS Keys with zero variance, we will reload the downsampled datasets that still contain the full set of 167 MACCS Keys. This will make future predictions easier because the pipeline will handle the zero-variance feature removal automatically.

As before, we need to define the configuration variables used for the grid search before creating and fitting the model.

In [ ]:
# GridSearchCV stands for Grid Search with Cross-Validation.
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import joblib

X_train = np.load("X_train_downsampled_MACCS167.npy")   # downsampled balanced set MACCS keys 167 values
y_train = np.load("y_train_downsampled_MACCS167.npy")   # downsampled balanced set activity values
X_test = np.load("X_test_MACCS167.npy")                 # test set MACCS keys 167 values
y_test = np.load("y_test_MACCS167.npy")                       # test set activity values

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)


# For each model configuration, calculate both ROC-AUC and balanced accuracy
scores = ["roc_auc", "balanced_accuracy"]

# Number of cross-validation folds
ncvs = 10

# Create ranges of values to test
max_depth_range = np.linspace(3, 7, num=5, dtype="int32")
min_samples_split_range = np.linspace(3, 7, num=5, dtype="int32")
min_samples_leaf_range = np.linspace(2, 6, num=5, dtype="int32")

# Create a pipeline that first removes zero-variance MACCS Keys, then fits a decision tree
pipe = Pipeline([
    ("var", VarianceThreshold(threshold=0.0)),
    ("clf", DecisionTreeClassifier(random_state=0))
])

# When using GridSearchCV with a pipeline, parameters are named as:
# pipeline_step_name__parameter_name
param_grid = {
    "clf__max_depth": max_depth_range,
    "clf__min_samples_split": min_samples_split_range,
    "clf__min_samples_leaf": min_samples_leaf_range
}

# Create the GridSearchCV object using the full pipeline
clf_DT_CV = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=ncvs,
    scoring=scores,
    refit="roc_auc",
    return_train_score=True
)

# Fit the grid search model
clf_DT_CV.fit(X_train, y_train)

# Print the best-performing hyperparameters
print("Best parameter set:", clf_DT_CV.best_params_)

# Save the full fitted GridSearchCV pipeline
joblib.dump(clf_DT_CV, "ds_maccs167_DTC_gridsearch_pipeline.joblib")

# Save the best parameters to use later
joblib.dump(clf_DT_CV.best_params_, "best_DT_params.joblib")

Now that we have saved the pipline, we can reload it and test a molecule:

In [ ]:
pipe = joblib.load("ds_maccs167_DTC_gridsearch_pipeline.joblib")
print("Pipeline loaded successfully!")
print()
# Create a fingerprint for a molecule
smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"  # aspirin
mol = Chem.MolFromSmiles(smiles)
fp = MACCSkeys.GenMACCSKeys(mol)

# prepare the array based on the fingerprint
X_query = np.array(fp).reshape(1, -1) # The classifier needs a 2D shape, but we have 1D list. We reshape (1, n_bits)
pred = pipe.predict(X_query)  # no manual masking needed
if pred == 0:
    print("Molecule is inactive for human aromatase.")
else:
    print("Molecule is active for human aromatase. Active may be agonist or antagonist in this model.")

# Returns the model's confidence for each possible class (e.g., [P(Inactive), P(Active)]).
probability = pipe.predict_proba(X_query)

print("Probabilities (Inactive, Active):", probability)

In [ ]:
# We can also loop through a list to predict multiple activities.

#new_smiles = "C1=CC=C(C(=C1)C2=NC(=NO2)C3=CC=NC=C3)Cl" #CID 65758 should be active
#new_smiles = "C1=CC(=CC=C1C2=COC3=CC(=CC(=C3C2=O)O)O)O" #CID 5280961 should be active
#new_smiles = "CN(C1CCN(CC1)C2=NC3=CC=CC=C3N2CC4=CC=C(C=C4)F)C5=NC=CC(=O)N5" #CID 65906 should be INactive
#new_smiles = "C1=CNC(=O)NC1=O" #CID 1174 should be INactive
#new_smiles = "C[C@H]1C[C@@H](C(=O)[C@@H](C1)[C@@H](CC2CC(=O)NC(=O)C2)O)C" #CID 6197 should be active

smiles_list = ["C1=CC=C(C(=C1)C2=NC(=NO2)C3=CC=NC=C3)Cl", "C1=CC(=CC=C1C2=COC3=CC(=CC(=C3C2=O)O)O)O", 
               "CN(C1CCN(CC1)C2=NC3=CC=CC=C3N2CC4=CC=C(C=C4)F)C5=NC=CC(=O)N5","C1=CNC(=O)NC1=O", 
               "C[C@H]1C[C@@H](C(=O)[C@@H](C1)[C@@H](CC2CC(=O)NC(=O)C2)O)C"]
fps = []
for s in smiles_list:
    mol = Chem.MolFromSmiles(s)
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros((fp.GetNumBits(),), dtype=int)
    ConvertToNumpyArray(fp, arr)
    fps.append(arr)

X_batch = np.array(fps)
preds = pipe.predict(X_batch)
print(preds)
preds = pipe.predict_proba(X_batch)
print(preds)

<div class="alert alert-block alert-warning">
<strong>Comparing Naive Bayes and Cross Validated Decision Tree Models</strong> Write your answers in the folloing raw cell.

1) For CID 65906, compare the prediction and probabilities for this molecule in the current cross validated decision tree vs the prediction and probabilities in the naive Bayes model from activity 08_03_ML_NB_activity.ipynb.  Did either model get this prediction correct? What do the probability values tell you about the reliability of the models?In your answer, identify whether each models was confidently correct, uncertain but correct, uncertain and incorrect, or confidently incorrect.

2) Of the three models created thus far(naïve bayes, decision tree, cross validated decision tree), which model do you have the most confidence in correctly identifying molecules that would be agonists or antagonists (active) at aromatase?